# 03 — Proportional Symbol Map
**Part 3 of 7** | GeoMetric Project

## Learning Objectives
- Scale symbols by area (not radius) — understand why this matters
- Handle overlap with transparency, sizing limits, and interactivity
- Compare proportional symbols vs choropleth for the same data

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import warnings; warnings.filterwarnings("ignore")
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster, HeatMap
from scripts.utils.config import PATHS, STYLE
from scripts.utils.map_utils import reproject_gdf, save_figure, add_map_annotations, scale_symbols, points_to_gdf

In [ ]:
world    = gpd.read_file(PATHS["processed"] / "master_world.gpkg")
airports = pd.read_csv(PATHS["processed"] / "airports_clean.csv")
top300   = airports.nlargest(300, "total_routes").dropna(subset=["lat","lon"])
print(f"Top 300 airports: min routes={top300['total_routes'].min()}, max={top300['total_routes'].max()}")

In [ ]:
# ── Why AREA scaling, not RADIUS scaling ───────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Radius Scaling vs Area Scaling — Perceptual Comparison", fontsize=13, fontweight="bold")

vals = pd.Series([10, 30, 100, 300, 1000])

# Radius scaling (misleading)
sizes_radius = (vals / vals.max() * 40) ** 2   # radius ∝ value → area ∝ value²
ax1.scatter(range(len(vals)), [1]*len(vals), s=sizes_radius,
            c="#d73027", alpha=0.7, edgecolors="white", linewidths=1)
ax1.set_xticks(range(len(vals)))
ax1.set_xticklabels([str(v) for v in vals])
ax1.set_yticks([])
ax1.set_title("❌ RADIUS scaling\nValue 1000 looks 10,000x bigger than value 10", color="red")

# Area scaling (correct)
sizes_area = scale_symbols(vals, min_size=20, max_size=2000, method="area")
ax2.scatter(range(len(vals)), [1]*len(vals), s=sizes_area,
            c="#2166ac", alpha=0.7, edgecolors="white", linewidths=1)
ax2.set_xticks(range(len(vals)))
ax2.set_xticklabels([str(v) for v in vals])
ax2.set_yticks([])
ax2.set_title("✅ AREA scaling\nValue 1000 looks 100x bigger than value 10 (correct)", color="green")

for ax in [ax1, ax2]:
    ax.set_xlabel("Data Value")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
save_figure(fig, PATHS["fig_part3"] / "radius_vs_area_scaling.png")
plt.show()

In [ ]:
# ── Static proportional symbol map ─────────────────────────
world_proj = reproject_gdf(world.copy(), "robinson")
ap_gdf  = points_to_gdf(top300, "lat", "lon")
ap_proj = reproject_gdf(ap_gdf, "robinson")
coords  = np.array([(g.x, g.y) for g in ap_proj.geometry])
sizes   = scale_symbols(ap_proj["total_routes"], min_size=5, max_size=1500, method="area")

fig, ax = plt.subplots(figsize=(18, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor(STYLE["ocean_color"])

world_proj.plot(ax=ax, color=STYLE["land_color"], linewidth=0.3, edgecolor=STYLE["boundary_color"])
sc = ax.scatter(coords[:,0], coords[:,1], s=sizes, c=ap_proj["total_routes"],
                cmap="plasma", alpha=0.65, edgecolors="white", linewidths=0.4, zorder=5)

cbar = fig.colorbar(sc, ax=ax, orientation="vertical", shrink=0.5, pad=0.01)
cbar.set_label("Total Routes", fontsize=9)

for ref in [20, 100, 300]:
    rs = scale_symbols(pd.Series([ref, top300["total_routes"].max()]), 5, 1500, "area")[0]
    ax.scatter([], [], s=rs, c="grey", alpha=0.6, label=f"{ref} routes")
ax.legend(title="Routes (area scale)", loc="lower left", fontsize=9, framealpha=0.85)
ax.set_axis_off()
add_map_annotations(ax, title="Global Airport Traffic — Proportional Symbol Map (Top 300)",
                    subtitle="Circle AREA proportional to number of routes | Colour intensity = route density",
                    source="OpenFlights.org", projection_name="robinson", year=2020)
save_figure(fig, PATHS["fig_part3"] / "map_proportional_static.png")
plt.show()

In [ ]:
# ── Interactive Folium map with tooltips ───────────────────
m = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron",
               prefer_canvas=True)

max_r = top300["total_routes"].max()

for _, row in top300.iterrows():
    radius = 4 + (row["total_routes"] / max_r) * 22
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius,
        color="#d73027", fill=True, fill_color="#d73027", fill_opacity=0.55,
        tooltip=folium.Tooltip(
            f"<div style='font-family:sans-serif;min-width:160px'>"
            f"<b style='font-size:13px'>{row['name']}</b><br>"
            f"<span style='color:#666'>{row['city']}, {row['country']}</span><br><hr>"
            f"🛫 <b>IATA:</b> {row['iata']}<br>"
            f"🔀 <b>Total Routes:</b> {int(row['total_routes'])}<br>"
            f"↗️  <b>Departures:</b> {int(row['departures'])}<br>"
            f"↙️  <b>Arrivals:</b> {int(row['arrivals'])}"
            f"</div>"
        )
    ).add_to(m)

legend = ("<div style='position:fixed;bottom:30px;left:30px;z-index:9999;
background:white;padding:12px 16px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.25);font-family:sans-serif'>
<b style='font-size:14px'>✈ Airport Traffic</b><br>
<span style='color:#d73027;font-size:22px'>●</span> Circle area ∝ routes<br>
<small>Source: OpenFlights.org</small></div>")
m.get_root().html.add_child(folium.Element(legend))

out = PATHS["interactive_folium"] / "airports_proportional.html"
m.save(str(out))
print(f"✅ Interactive map saved: {out}")
m

In [ ]:
# ── Proportional symbols vs choropleth comparison ──────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("Proportional Symbols vs Choropleth — Same Variable, Different Stories",
             fontsize=14, fontweight="bold")

# Choropleth: routes aggregated by country
country_routes = airports.groupby("country")["total_routes"].sum().reset_index()
world_r = world.merge(country_routes, left_on="country_name", right_on="country", how="left")
world_rp = reproject_gdf(world_r, "robinson")

world_rp[world_rp["total_routes"].isna()].plot(ax=ax1, color="#ddd", linewidth=0.2, edgecolor="#bbb")
world_rp.dropna(subset=["total_routes"]).plot(
    column="total_routes", ax=ax1, scheme="Quantiles", k=5, cmap="Blues",
    legend=True, legend_kwds={"title":"Routes by country","loc":"lower left","fontsize":8},
    linewidth=0.2, edgecolor="#888", missing_kwds={"color":"#ddd"}
)
ax1.set_axis_off(); ax1.set_facecolor(STYLE["ocean_color"])
ax1.set_title("Choropleth (aggregated by country)
⚠ Fills entire country — implies uniform distribution",
              fontsize=10)

# Proportional symbols
world_proj.plot(ax=ax2, color=STYLE["land_color"], linewidth=0.2, edgecolor="#aaa")
ax2.scatter(coords[:,0], coords[:,1], s=sizes, c="#d73027", alpha=0.6,
            edgecolors="white", linewidths=0.3, zorder=5)
ax2.set_axis_off(); ax2.set_facecolor(STYLE["ocean_color"])
ax2.set_title("Proportional Symbols (exact locations)
✅ Places traffic at the actual airport",
              fontsize=10)

plt.tight_layout()
save_figure(fig, PATHS["fig_part3"] / "choropleth_vs_proportional.png")
plt.show()